# Pose Landmarks Detection with MediaPipe Tasks

This notebook shows you how to use MediaPipe Tasks Python API to detect pose landmarks from images.

## Preparation

Let's start with installing MediaPipe.  
Then download an off-the-shelf model bundle. Check out the [MediaPipe documentation](https://developers.google.com/mediapipe/solutions/vision/pose_landmarker#models) for more information about this model bundle.

In [ ]:
!pip install -q mediapipe

In [ ]:
!wget -O pose_landmarker.task -q https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_heavy/float16/1/pose_landmarker_heavy.task

In [ ]:
from google.colab import drive
from google.colab.patches import cv2_imshow
import cv2

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Visualization utilities

In [ ]:
import numpy as np
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles
from mediapipe.tasks.python import vision


def draw_landmarks_on_image(rgb_image, detection_result):
  pose_landmarks_list = detection_result.pose_landmarks
  annotated_image = np.copy(rgb_image)

  pose_landmark_style = drawing_styles.get_default_pose_landmarks_style()
  pose_connection_style = drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=3, circle_radius=4)

  for pose_landmarks in pose_landmarks_list:
    drawing_utils.draw_landmarks(
        image=annotated_image,
        landmark_list=pose_landmarks,
        connections=vision.PoseLandmarksConnections.POSE_LANDMARKS,
        landmark_drawing_spec=pose_landmark_style,
        connection_drawing_spec=pose_connection_style)

  return annotated_image

## 🖼️ Get LandMarks for Image

In [ ]:
img = cv2.imread("/content/drive/MyDrive/Dance App/dance_pose_sample.png")

numpy_image = img.copy()

if numpy_image.shape[2] == 4:
    numpy_image = cv2.cvtColor(numpy_image, cv2.COLOR_RGBA2RGB)

## Running inference and visualizing the results

The final step is to run pose landmark detection on your selected image. This involves creating your PoseLandmarker object, loading your image, running detection, and finally, the optional step of displaying the image with visualizations.

Check out the [MediaPipe documentation](https://developers.google.com/mediapipe/solutions/vision/pose_landmarker/python) to learn more about configuration options that this solution supports.


In [ ]:
# STEP 1: Import the necessary modules.
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# STEP 2: Create an PoseLandmarker object.
base_options = python.BaseOptions(model_asset_path='pose_landmarker.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=True)
detector = vision.PoseLandmarker.create_from_options(options)

# STEP 3: Load the input image.
image = mp.Image.create_from_file("/content/drive/MyDrive/Dance App/dance_pose_sample.png")

# STEP 4: Detect pose landmarks from the input image.
detection_result = detector.detect(image)

# STEP 5: Process the detection result. In this case, visualize it.
annotated_image = draw_landmarks_on_image(numpy_image, detection_result)

# 4. Display the image
cv2_imshow(cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR))

Visualize the pose segmentation mask.

## 📽️ Get Landmarks for video

In [66]:
INPUT_VIDEO  = "/content/drive/MyDrive/Dance App/dance_sample_0_6_seconds.mp4"
OUTPUT_VIDEO = "/content/drive/MyDrive/Dance App/dance_sample_0_6_seconds_landmarks.mp4"

# Helper: blend the segmentation mask on top of the frame
MASK_COLOR_RGB = np.array([0, 200, 255], dtype=np.float32)   # warm cyan
MASK_ALPHA     = 0.35

def overlay_mask(rgb, detection_result):
    if not detection_result.segmentation_masks:
        return rgb
    seg = detection_result.segmentation_masks[0].numpy_view()        # may be (H,W) or (H,W,1)
    seg = np.squeeze(seg)                                             # -> (H,W)
    m   = np.clip(seg, 0.0, 1.0)[..., None]                           # (H,W,1)
    out = rgb.astype(np.float32) * (1 - MASK_ALPHA * m) + MASK_COLOR_RGB * (MASK_ALPHA * m)
    return out.astype(np.uint8)

In [67]:
# STEP 1: Imports
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# STEP 2: PoseLandmarker in VIDEO mode, with segmentation masks
base_options = python.BaseOptions(model_asset_path='pose_landmarker.task')
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    output_segmentation_masks=True,
    running_mode=vision.RunningMode.VIDEO,
)
detector = vision.PoseLandmarker.create_from_options(options)

# STEP 3: Open input video
cap = cv2.VideoCapture(INPUT_VIDEO)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {INPUT_VIDEO}")

fps          = cap.get(cv2.CAP_PROP_FPS)
width        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height       = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"fps={fps:.3f}  size={width}x{height}  frames={total_frames}")

# STEP 4: Open writer with same fps/resolution
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (width, height))

# STEP 5: Stream frames
frame_idx = 0
while True:
    ok, frame_bgr = cap.read()
    if not ok:
        break

    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

    # Monotonically-increasing timestamp derived from frame index
    timestamp_ms = int(round(frame_idx * 1000.0 / fps))

    detection_result = detector.detect_for_video(mp_image, timestamp_ms)

    annotated_rgb = overlay_mask(frame_rgb, detection_result)                    # mask first
    annotated_rgb = draw_landmarks_on_image(annotated_rgb, detection_result)     # landmarks on top

    writer.write(cv2.cvtColor(annotated_rgb, cv2.COLOR_RGB2BGR))

    frame_idx += 1
    if frame_idx % 30 == 0:
        print(f"processed {frame_idx}/{total_frames}")

cap.release()
writer.release()
detector.close()
print(f"Done. Wrote {frame_idx} frames to {OUTPUT_VIDEO}")

fps=30.000  size=1080x1920  frames=179
processed 30/179
processed 60/179
processed 90/179
processed 120/179
processed 150/179
Done. Wrote 179 frames to /content/drive/MyDrive/Dance App/dance_sample_0_6_seconds_landmarks.mp4
